In [1]:
import firebase_admin
from firebase_admin import credentials, storage
from firebase_admin import db
import pandas as pd
import os
import dotenv
dotenv.load_dotenv()

True

# Firebase Initialization

In [2]:
service_account_info ={
    "type": os.getenv('FIREBASE_TYPE'),
    "project_id": os.getenv('FIREBASE_PROJECT_ID'),
    "private_key_id": os.getenv('FIREBASE_PRIVATE_KEY_ID'),
    "private_key": os.getenv('FIREBASE_PRIVATE_KEY'),
    "client_email": os.getenv('FIREBASE_CLIENT_EMAIL'),
    "client_id": os.getenv('FIREBASE_CLIENT_ID'),
    "auth_uri": os.getenv('FIREBASE_AUTH_URI'),
    "token_uri": os.getenv('FIREBASE_TOKEN_URI'),
    "auth_provider_x509_cert_url": os.getenv('FIREBASE_AUTH_PROVIDER_X509_CERT_URL'),
    "client_x509_cert_url": os.getenv('FIREBASE_CLIENT_X509_CERT_URL'),
    "universe_domain": os.getenv('FIREBASE_UNIVERSE_DOMAIN')
  }
  

In [3]:
cred = credentials.Certificate(service_account_info)
firebase_admin.initialize_app(cred,{
    'storageBucket': 'ss-management-chatbot.firebasestorage.app',
     'databaseURL': 'https://ss-management-chatbot-default-rtdb.firebaseio.com/'
})

In [4]:
bucket = storage.bucket()

# Upload Data

In [5]:
image_folder_path = './data/images/'

In [22]:
students_collection = db.reference('students')

In [24]:
df = pd.read_json('data/students.json')

df.head(2)

,full_name,registration_number,email,region,description,image_path,year,department,status
0,Hiba Mansour,S12255237,s12255237@stu.najah.edu,Palestine,Computer science student with a passion for da...,hiba_mansour.jpg,3rd Year,Computer Science,active
1,Mai Shelbayeh,N/A,Maishelbayeh@icloud.com,Palestine,AI enthusiast with interest in mobile app deve...,mai_shelbayeh.jpg,1st Year,Software Engineering,active


In [25]:
def upload_image(bucket, image_path):
    image_name = image_path.split('/')[-1]
    blob = bucket.blob(f'students_images/{image_name}')
    # Upload image
    blob.upload_from_filename(image_path)
    # Make the image publicly accessible and get its URL
    blob.make_public()
    return blob.public_url

In [27]:
for index, row in df.iterrows():
    print(index, row['full_name'])
    
    image_path = os.path.join(image_folder_path,row['image_path'])
    
    image_url = upload_image(bucket,image_path)
    product_data = row.to_dict()
    product_data.pop('image_path')
    product_data['image_url']= image_url
    
    # Add to Firestore
    students_collection.push().set(product_data)
    

0 Hiba Mansour
1 Mai Shelbayeh
2 Orwa Jabali
3 Raghad Hethnawi
4 Aseel Omar
5 Raghad Sayeh
6 Hanan Rayyan
7 Marya Amouri
8 Tabarak Najjar
9 Rawan Awaysa
10 Laith Jabali
11 Sara Taher
12 Bahaa Shanaa
13 Tasbeeh Takrori
14 Walid AbuZainah
